# 02 — AI-Assisted Quality Evaluators

**What you'll learn:**
- How LLM-as-judge evaluators score meaning, not just wording
- Configuring a judge model (`model_config`)
- Groundedness, relevance, coherence, fluency, retrieval, similarity, and the composite `QAEvaluator`

---

AI-assisted evaluators use a **judge model** (an Azure OpenAI deployment) to score
outputs on a 1–5 scale with reasoning. They understand paraphrase and semantics —
the gap the NLP metrics can't close — at the cost of API calls to the judge.

In [1]:
import os
import site
import shutil
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

# --------------------------------------------------------------------------
# Reasoning-model compatibility (gpt-5.x / o-series judge models)
# --------------------------------------------------------------------------
# Reasoning models reject `max_tokens` (they require `max_completion_tokens`) and
# only allow the default temperature. azure-ai-evaluation runs evaluators in
# spawned worker processes, so the fix must load in every worker: we copy the
# workshop's `sitecustomize.py` shim into site-packages (auto-imported by every
# Python process) and also apply it here in the kernel. It is a no-op for classic
# models (gpt-4o etc.), so it is always safe to run.
_shim_src = Path("../sitecustomize.py")
if _shim_src.exists():
    _shim_dst = Path(site.getsitepackages()[0]) / "sitecustomize.py"
    if not _shim_dst.exists() or _shim_dst.read_text() != _shim_src.read_text():
        shutil.copyfile(_shim_src, _shim_dst)
    exec(_shim_src.read_text())

# Judge model config for AI-assisted evaluators (notebooks 02, 04, 05).
# Uses az login / DefaultAzureCredential when AZURE_OPENAI_API_KEY is blank.
model_config = {
    "azure_endpoint": os.environ["AZURE_OPENAI_ENDPOINT"],
    "azure_deployment": os.environ["AZURE_OPENAI_DEPLOYMENT"],
    "api_version": os.environ.get("AZURE_OPENAI_API_VERSION", "2024-10-21"),
}
if os.environ.get("AZURE_OPENAI_API_KEY"):
    model_config["api_key"] = os.environ["AZURE_OPENAI_API_KEY"]

# Foundry project (notebooks 03, 06) — enables cloud logging + safety evaluators.
azure_ai_project = os.environ.get("AZURE_AI_PROJECT_ENDPOINT")

DATA_DIR = "../data"
print("Judge deployment:", model_config["azure_deployment"])

Judge deployment: gpt-5.4-mini


## The judge model

Every AI-assisted evaluator takes a `model_config` pointing at your judge deployment.
We built `model_config` in the setup cell from your `.env`. Keep the judge model
**fixed** across runs so scores stay comparable.

## Quality evaluators

| Evaluator | Question it answers | Needs `context`? | Needs `ground_truth`? |
|-----------|---------------------|:---:|:---:|
| `GroundednessEvaluator` | Is the answer supported by the context? | yes | no |
| `RelevanceEvaluator` | Does the answer address the query? | no | no |
| `CoherenceEvaluator` | Is it logically consistent and readable? | no | no |
| `FluencyEvaluator` | Is the language natural and correct? | no | no |
| `RetrievalEvaluator` | Did retrieval surface the right context? | yes | no |
| `SimilarityEvaluator` | How close is it to the reference answer? | no | yes |

In [2]:
from azure.ai.evaluation import (
    evaluate,
    GroundednessEvaluator,
    RelevanceEvaluator,
    CoherenceEvaluator,
    FluencyEvaluator,
    RetrievalEvaluator,
    SimilarityEvaluator,
)

quality_evaluators = {
    "groundedness": GroundednessEvaluator(model_config),
    "relevance": RelevanceEvaluator(model_config),
    "coherence": CoherenceEvaluator(model_config),
    "fluency": FluencyEvaluator(model_config),
    "retrieval": RetrievalEvaluator(model_config),
    "similarity": SimilarityEvaluator(model_config),
}

In [3]:
# grounded-context.jsonl has query/context/response/ground_truth for every row.
result = evaluate(
    data=f"{DATA_DIR}/grounded-context.jsonl",
    evaluators=quality_evaluators,
)
result["metrics"]

Version mismatch between change and config responses.No configurations updated.


2026-07-21 10:20:52 -0400 137698937509568 execution.bulk     INFO     Finished 1 / 6 lines.
2026-07-21 10:20:52 -0400 137698937509568 execution.bulk     INFO     Average execution time for completed lines: 8.91 seconds. Estimated time for incomplete lines: 44.55 seconds.
2026-07-21 10:20:52 -0400 137698929116864 execution.bulk     INFO     Finished 1 / 6 lines.
2026-07-21 10:20:52 -0400 137698929116864 execution.bulk     INFO     Average execution time for completed lines: 8.93 seconds. Estimated time for incomplete lines: 44.65 seconds.
2026-07-21 10:20:52 -0400 137698929116864 execution.bulk     INFO     Finished 2 / 6 lines.
2026-07-21 10:20:52 -0400 137698929116864 execution.bulk     INFO     Average execution time for completed lines: 4.47 seconds. Estimated time for incomplete lines: 17.88 seconds.
2026-07-21 10:20:52 -0400 137698895546048 execution.bulk     INFO     Finished 1 / 6 lines.
2026-07-21 10:20:52 -0400 137698895546048 execution.bulk     INFO     Average execution time

Aggregated metrics for evaluator is not a dictionary will not be logged as metrics


======= Run Summary =======

Run name: "groundedness_20260721_142043_665066"
Run status: "Completed"
Start time: "2026-07-21 14:20:43.665066+00:00"
Duration: "0:00:09.853720"

2026-07-21 10:20:53 -0400 137698912331456 execution.bulk     INFO     Finished 3 / 6 lines.
2026-07-21 10:20:53 -0400 137698912331456 execution.bulk     INFO     Average execution time for completed lines: 3.32 seconds. Estimated time for incomplete lines: 9.96 seconds.
2026-07-21 10:20:53 -0400 137698912331456 execution.bulk     INFO     Finished 4 / 6 lines.
2026-07-21 10:20:53 -0400 137698912331456 execution.bulk     INFO     Average execution time for completed lines: 2.49 seconds. Estimated time for incomplete lines: 4.98 seconds.


Aggregated metrics for evaluator is not a dictionary will not be logged as metrics


======= Run Summary =======

Run name: "relevance_20260721_142043_671403"
Run status: "Completed"
Start time: "2026-07-21 14:20:43.671403+00:00"
Duration: "0:00:10.013967"

2026-07-21 10:20:53 -0400 137698903938752 execution.bulk     INFO     Finished 3 / 6 lines.
2026-07-21 10:20:53 -0400 137698903938752 execution.bulk     INFO     Average execution time for completed lines: 3.38 seconds. Estimated time for incomplete lines: 10.14 seconds.
2026-07-21 10:20:53 -0400 137698903938752 execution.bulk     INFO     Finished 4 / 6 lines.
2026-07-21 10:20:53 -0400 137698903938752 execution.bulk     INFO     Average execution time for completed lines: 2.54 seconds. Estimated time for incomplete lines: 5.08 seconds.
2026-07-21 10:20:53 -0400 137698912331456 execution.bulk     INFO     Finished 5 / 6 lines.
2026-07-21 10:20:53 -0400 137698912331456 execution.bulk     INFO     Average execution time for completed lines: 2.03 seconds. Estimated time for incomplete lines: 2.03 seconds.
2026-07-21 10

... (5 outputs left out)

2026-07-21 10:21:33 -0400 137696181745344 execution.bulk     INFO     Finished 6 / 6 lines.
2026-07-21 10:21:33 -0400 137696181745344 execution.bulk     INFO     Average execution time for completed lines: 6.47 seconds. Estimated time for incomplete lines: 0.0 seconds.


Aggregated metrics for evaluator is not a dictionary will not be logged as metrics


======= Run Summary =======

Run name: "qa_20260721_142054_715664"
Run status: "Completed"
Start time: "2026-07-21 14:20:54.715664+00:00"
Duration: "0:00:38.834509"

======= Combined Run Summary (Per Evaluator) =======

{
    "qa": {
        "status": "Completed",
        "duration": "0:00:38.834509",
        "completed_lines": 6,
        "failed_lines": 0,
        "log_path": null,
        "per_line_errors": {},
        "error_message": null,
        "error_code": null
    }
}




Version mismatch between change and config responses.No configurations updated.


In [4]:
import pandas as pd

# Each AI-assisted evaluator returns a score plus a natural-language reason.
rows = pd.DataFrame(result["rows"])
rows[[c for c in rows.columns if "groundedness" in c]].head()

,outputs.groundedness.groundedness,outputs.groundedness.groundedness_score,outputs.groundedness.groundedness_passed,outputs.groundedness.groundedness_result,outputs.groundedness.groundedness_reason,outputs.groundedness.groundedness_status,outputs.groundedness.groundedness_threshold,outputs.groundedness.groundedness_properties
0,4.0,4.0,True,pass,Let's think step by step: The response accurat...,completed,3,"{'prompt_tokens': 1165, 'completion_tokens': 9..."
1,4.0,4.0,True,pass,Let's think step by step: The response accurat...,completed,3,"{'prompt_tokens': 1166, 'completion_tokens': 1..."
2,4.0,4.0,True,pass,Let's think step by step: The response correct...,completed,3,"{'prompt_tokens': 1146, 'completion_tokens': 7..."
3,5.0,5.0,True,pass,Let's think step by step: The response is dire...,completed,3,"{'prompt_tokens': 1163, 'completion_tokens': 7..."
4,5.0,5.0,True,pass,Let's think step by step: The response is dire...,completed,3,"{'prompt_tokens': 1175, 'completion_tokens': 8..."


## The composite `QAEvaluator`

Running every quality metric by hand is verbose. `QAEvaluator` bundles groundedness,
relevance, coherence, fluency, similarity, and F1 into a single evaluator.

In [5]:
from azure.ai.evaluation import QAEvaluator

qa_result = evaluate(
    data=f"{DATA_DIR}/grounded-context.jsonl",
    evaluators={"qa": QAEvaluator(model_config)},
)
qa_result["metrics"]

{'qa.f1_score': 0.48923761423761425,
 'qa.f1_score_score': 0.48923761423761425,
 'qa.f1_score_passed': 0.3333333333333333,
 'qa.coherence': 4.166666666666667,
 'qa.coherence_score': 4.166666666666667,
 'qa.coherence_passed': 1.0,
 'qa.similarity': 4.666666666666667,
 'qa.similarity_score': 4.666666666666667,
 'qa.similarity_passed': 1.0,
 'qa.fluency': 3.5,
 'qa.fluency_score': 3.5,
 'qa.fluency_passed': 1.0,
 'qa.relevance': 4.0,
 'qa.relevance_score': 4.0,
 'qa.relevance_passed': 1.0,
 'qa.groundedness': 4.833333333333333,
 'qa.groundedness_score': 4.833333333333333,
 'qa.groundedness_passed': 1.0,
 'qa.binary_aggregate': 1.0}

## Recap

- AI-assisted evaluators judge *meaning* with an LLM and return score + reason.
- Pass a fixed `model_config` so results stay comparable across runs.
- `QAEvaluator` is a convenient all-in-one quality bundle.

**Next:** [03 — Risk & safety](./03-risk-and-safety.ipynb) — evaluators backed by the
Azure AI Foundry safety service.